# Laboratorio cuantitativo de criptoactivos — Python

De las velas de Binance al modelo, en nueve pasos. Cada paso tiene su módulo en
`cryptoquant/laboratorio/`; aquí se ve el código y se discute el resultado.

| Paso | Tema | Dónde |
|---|---|---|
| 1 | Entorno: DataFrame, velas de Binance, gráfico de velas, CSV | aquí |
| 2 | Procesamiento con dplyr y ggplot2 | `laboratorio/r/` |
| 3 | Preparación de variables | aquí |
| 4 | Modelamiento cuantitativo | aquí |
| 5 | Series de tiempo | aquí y en R |
| 6 | Volatilidad (GARCH) | aquí y en R |
| 7 | Machine learning (random forest) | aquí |
| 8 | Monte Carlo | aquí |
| 9 | Control de volatilidad: la idea de la tesis | aquí y en R |

El paso 1 exporta el CSV que lee la versión en R.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# El notebook vive en laboratorio/python/: la raiz del proyecto esta dos niveles arriba.
RAIZ = Path.cwd().resolve().parents[1]
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from cryptoquant.laboratorio import (aprendizaje, control, cuantitativo, datos,
                                     montecarlo, series, variables, volatilidad)

pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams["figure.figsize"] = (11, 4.5)

## 1. Entorno en Python

### Una vela, a mano
Binance publica sus velas gratis y sin clave en data-api.binance.vision, que funciona también desde Colab. `ccxt` las devuelve como listas
`[marca de tiempo en ms, apertura, máximo, mínimo, cierre, volumen]`.

In [ ]:
import ccxt

binance = ccxt.binance()
# api.binance.com bloquea las IP de EE. UU. (Colab): se usa el dominio de datos publicos.
binance.urls["api"]["public"] = "https://data-api.binance.vision/api/v3"
crudo = binance.fetch_ohlcv("BTC/USDT", timeframe="1h", limit=5)
crudo

Convertirlas en un `DataFrame` de pandas con índice temporal:

In [ ]:
df = pd.DataFrame(crudo, columns=["ts", "open", "high", "low", "close", "volume"])
df["fecha"] = pd.to_datetime(df["ts"], unit="ms", utc=True)
df = df.set_index("fecha").drop(columns="ts")
df

La última fila es la vela **en curso**: su cierre es el precio de este
instante y va a cambiar. Nunca debe usarse como dato cerrado.

### El histórico completo
Para varios años hay que paginar (Binance entrega 1000 velas por petición).
`datos.cargar_velas` lo hace, descarta la vela en curso y exporta el CSV. Si
el CSV ya existe lo reutiliza; `descargar=True` fuerza una descarga nueva.

In [ ]:
velas, origen = datos.cargar_velas("BTC", "1h", desde="2022-01-01")
print(origen, velas.shape)
velas.tail()

In [ ]:
velas.info()
velas.describe()

### Gráfico de velas
Verde si cerró por encima de la apertura, rojo si por debajo. Las mechas
marcan máximo y mínimo.

In [ ]:
diario = datos.a_diario(velas)   # agrega las horas en velas diarias UTC
datos.grafico_velas(diario, ultimas=120, titulo="BTC/USDT diario")
plt.show()

In [ ]:
ruta = datos.ruta_csv("BTC/USDT", "1h")
print(ruta)
print(ruta.read_text().splitlines()[:3])

## 3. Preparación de variables

Cada fila `t` describe lo que se sabía al cierre de `t`. Solo `retorno_futuro`
y `sube` miran a `t+1`: son el objetivo, nunca una variable explicativa.

Las columnas de nivel de precio (`sma_20`, `bb_sup`) sirven para graficar. Al
modelo le pasamos versiones adimensionales (`dist_sma_20`, `bb_pct`), porque un
árbol entrenado con precios de 2022 no sabe qué hacer con los de 2026.

In [ ]:
v = variables.preparar_variables(diario)
v[["close", "retorno", "retorno_l1", "sma_20", "rsi_14", "macd", "macd_senal",
   "bb_inf", "bb_sup", "vol_20", "retorno_futuro", "sube"]].tail(8)

In [ ]:
variables.VARIABLES_MODELO

**Comprobación de causalidad.** Si una variable mira al futuro, al recortar
la serie cambian sus valores pasados. Este es el test más importante del
laboratorio (también está en `tests/test_laboratorio.py`).

In [ ]:
k = len(diario) - 100
corta = variables.preparar_variables(diario.iloc[:k])
cols = [c for c in v.columns if c not in ("retorno_futuro", "sube")]
pd.testing.assert_frame_equal(v[cols].iloc[:k], corta[cols], check_exact=False, atol=1e-10)
print("Ninguna variable explicativa mira al futuro.")

## 4. Modelamiento cuantitativo

### Colas gruesas
Si los retornos fueran normales, un día de más de 4 desviaciones pasaría
una vez cada ~40 años.

In [ ]:
ret = v["retorno"].dropna()
cuantitativo.distribucion_retornos(ret)

In [ ]:
cuantitativo.tabla_colas(ret)

In [ ]:
from scipy import stats

z = (ret - ret.mean()) / ret.std()
fig, (a1, a2) = plt.subplots(1, 2)
a1.hist(z, bins=120, density=True, alpha=0.6, label="observado")
xs = np.linspace(-8, 8, 400)
a1.plot(xs, stats.norm.pdf(xs), label="normal")
a1.set_yscale("log"); a1.set_ylim(1e-5, 1); a1.legend()
stats.probplot(z, dist="norm", plot=a2)
plt.show()

En escala logarítmica la normal cae como una parábola; los datos, mucho más
despacio. En el QQ, los extremos se separan de la recta: esas son las colas.

### Estadística de señales
Acierto y retorno medio después de cada señal, **frente a la base**: la misma
dirección aplicada a todos los días. Con 24 contrastes, alguno sale
significativo por azar; el umbral honesto es `0.05 / 24`.

In [ ]:
s = cuantitativo.estadistica_senales(v)
s[s["horizonte"] == 5]

In [ ]:
print("Significativas tras Bonferroni:", (s["p_valor"] < 0.05 / len(s)).sum(), "de", len(s))

## 5. Series de tiempo

### Por qué retornos y no precios
ADF: la hipótesis nula es raíz unitaria (no estacionaria).

In [ ]:
series.precio_frente_a_retorno(diario["close"])

### Autocorrelación
La dirección de mañana casi no depende de la de hoy, pero su **magnitud** sí.
Esa es la idea de todo lo que sigue: la volatilidad se puede prever; la
dirección, apenas.

In [ ]:
acf_tabla, ljung = series.autocorrelacion(ret)
display(ljung)
acf_tabla.head(10)

In [ ]:
fig, ax = plt.subplots()
ax.bar(acf_tabla.index - 0.2, acf_tabla["retorno"], width=0.4, label="retorno")
ax.bar(acf_tabla.index + 0.2, acf_tabla["abs_retorno"], width=0.4, label="|retorno|")
b = acf_tabla["banda_95"].iloc[0]
ax.axhspan(-b, b, color="grey", alpha=0.2); ax.legend(); plt.show()

### Patrones por hora y día (UTC)

In [ ]:
ret_h = np.log(velas["close"]).diff().dropna()
cal = series.patrones_calendario(ret_h)
for k, t in cal.items():
    print(k, t.attrs)
fig, (a1, a2) = plt.subplots(1, 2)
a1.bar(cal["hora"].index, cal["hora"]["volatilidad"]); a1.set_title("volatilidad por hora")
a2.bar(cal["dia"].index, cal["dia"]["volatilidad"]); a2.set_title("volatilidad por día")
a2.tick_params(axis="x", rotation=60); plt.show()

### ARIMA básico
Elegido por AIC con el 80% inicial y evaluado a un paso sobre el 20% final,
contra la previsión trivial de retorno cero.

In [ ]:
series.arima_basico(ret)

## 6. Volatilidad

$\sigma^2_{t+1} = \omega + \alpha\, r_t^2 + \beta\, \sigma^2_t$

In [ ]:
volatilidad.agrupamiento(ret)

In [ ]:
g = volatilidad.pronostico_garch(ret)
{k: val for k, val in g.items() if k != "sigma_condicional"}

In [ ]:
(g["sigma_condicional"] * np.sqrt(365)).plot(label="GARCH(1,1)", lw=0.9)
(v["vol_20"] * np.sqrt(365)).plot(label="móvil 20 días", lw=0.9, alpha=0.7)
plt.legend(); plt.title("Volatilidad anualizada"); plt.show()

¿Predice mejor que una desviación móvil? Parámetros estimados con el 70%
inicial y congelados; se mide con QLIKE (menor es mejor).

In [ ]:
volatilidad.evaluar_garch(ret)

### Del pronóstico a la operación
Stop a 2 sigmas previstas; tamaño tal que, si salta el stop, se pierde el 1%
de la cuenta. Con más volatilidad: stop más lejos, posición más pequeña,
misma pérdida.

In [ ]:
volatilidad.plan_operacion(precio=float(diario["close"].iloc[-1]), sigma=g["sigma_manana"],
                           capital=10_000, riesgo_por_operacion=0.01, k_stop=2, riesgo_beneficio=2)

## 7. Machine learning

Random forest que clasifica si mañana sube. Se entrena con el 70% más antiguo,
se deja una vela de embargo y se prueba con el resto. Dos líneas base: decir
siempre la clase mayoritaria del entrenamiento, y repetir la dirección de hoy.

In [ ]:
rf = aprendizaje.entrenar_bosque(v)
print("prueba desde", rf.inicio_prueba.date(), "| p-valor frente a la mejor base:",
      round(rf.p_valor_vs_base, 3))
rf.metricas

In [ ]:
imp = rf.importancia
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(imp.index[::-1], imp["permutacion_auc"][::-1], xerr=imp["permutacion_sd"][::-1])
ax.axvline(0, color="grey"); ax.set_title("Importancia por permutación (caída de AUC)")
plt.show()

La importancia **por impureza** siempre reparte algo entre todas las
variables, aunque ninguna sirva. La de **permutación** mide cuánto empeora el
modelo en prueba al desordenar una variable: si ronda cero o es negativa, la
variable no aporta fuera de muestra.

## 8. Monte Carlo

### Trayectorias de precio
Remuestreo por bloques de 5 días de los retornos históricos.

In [ ]:
caminos = montecarlo.trayectorias_precio(ret, float(diario["close"].iloc[-1]), dias=90, n=5000)
q = np.percentile(caminos, [5, 25, 50, 75, 95], axis=0)
t = np.arange(caminos.shape[1])
plt.fill_between(t, q[0], q[4], alpha=0.2, label="5-95%")
plt.fill_between(t, q[1], q[3], alpha=0.35, label="25-75%")
plt.plot(t, q[2], label="mediana"); plt.legend(); plt.show()
montecarlo.resumen_trayectorias(caminos)

### Simulación de la cuenta
Cambie `acierto`, `rb` (riesgo-beneficio 1:rb) y `riesgo` por operación. Con
`acierto = 1/(1+rb)` la esperanza es cero, y aun así la mediana termina
perdiendo: con fracción fija, las pérdidas pesan más que las ganancias.

In [ ]:
acierto, rb, riesgo = 0.45, 2.0, 0.01
curvas = montecarlo.simular_cuenta(acierto, rb, riesgo, n_operaciones=200)
for c in curvas[:100]:
    plt.plot(c, lw=0.5, alpha=0.3)
plt.plot(np.median(curvas, axis=0), color="black", lw=1.5)
plt.yscale("log"); plt.show()
montecarlo.resumen_cuenta(curvas, acierto, rb, riesgo)

In [ ]:
# Mismo acierto y R:B, arriesgando 10% por operacion
curvas10 = montecarlo.simular_cuenta(acierto, rb, 0.10, n_operaciones=200)
montecarlo.resumen_cuenta(curvas10, acierto, rb, 0.10)

## 9. Control de volatilidad: la idea de la tesis

La dirección apenas se puede prever (pasos 5 y 7); la volatilidad sí
(paso 6). En lugar de apostar por la dirección, se ajusta cuánto se
invierte para que el riesgo quede cerca de un objetivo:

$\text{exposición}_t = \min(1,\ \sigma^* / \hat\sigma_{t+1})$

con $\hat\sigma$ prevista por EWMA ($\lambda = 0{,}94$). El resto queda en
efectivo. La exposición decidida al cierre de hoy se aplica al rendimiento de
mañana, y se paga un 0,15 % por cada unidad que cambia: nada mira al futuro.

In [ ]:
objetivo = 0.15   # volatilidad anual que se quiere soportar; pruebe 0.30
ctl = control.control_volatilidad(ret, objetivo_anual=objetivo)
tabla_ctl = control.resumen(ctl)
print(f"Parte invertida, de media: {tabla_ctl.attrs['exposicion_media']:.0%}")
tabla_ctl

In [ ]:
from matplotlib.ticker import FuncFormatter

capital = 10_000 * (1 + ctl[["estrategia", "pasiva", "pasiva_igual_vol"]]).cumprod()
capital.columns = ["Control de volatilidad", "Comprar y mantener",
                   "Misma volatilidad (ex post)"]
fig, (a1, a2) = plt.subplots(2, 1, sharex=True, figsize=(11, 7),
                             gridspec_kw={"height_ratios": [2, 1]})
capital.plot(ax=a1, logy=True, lw=0.9, title="El riesgo lo fija usted, no el mercado")
dolares = FuncFormatter(lambda y, _: f"{y:,.0f}")
a1.yaxis.set_major_formatter(dolares); a1.yaxis.set_minor_formatter(dolares)
a1.set_ylabel("10 000 dólares invertidos (escala log)")
ctl["exposicion"].plot(ax=a2, lw=0.8, color="grey")
a2.set_ylabel("parte invertida"); a2.set_ylim(0, 1.05); a2.set_xlabel("")
plt.show()

El control no gana más que comprar y mantener: arriesga mucho menos. Cuando
el mercado se agita, la parte invertida baja sola, y la caída máxima pasa de
la de bitcoin a una fracción de ella.

La comparación justa es con la pasiva **a igual volatilidad**: una parte fija
en bitcoin y el resto en efectivo. Pero esa referencia necesitó conocer la
volatilidad de toda la muestra para elegir su fracción; el control llegó al
mismo riesgo sin conocer el futuro. Es el resultado de la tesis: el valor está
en controlar el riesgo, no en predecir el precio.

## Siguiente paso
`python -m cryptoquant laboratorio` ejecuta todo lo anterior y deja las tablas
y gráficos en `reports/laboratorio/`. La versión en R está en
`laboratorio/r/laboratorio.Rmd`.